### Подготовка окружения

Добавляем путь к папке code для получения доступа к классам и функциям thinkdsp.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd()
code_path = current_path.parent / 'code'
sys.path.append(str(code_path))

In [2]:
import numpy as np

PI2 = 2 * np.pi

### Эталонная реализация DFT

Матричное умножение $O(N^2)$ из книги — используем для сравнения.

In [3]:
def dft(ys):
    N = len(ys)
    ts = np.arange(N) / N
    freqs = np.arange(N)
    args = np.outer(ts, freqs)
    M = np.exp(1j * PI2 * args)
    amps = M.conj().transpose().dot(ys)
    return amps

### Тестовый сигнал

Небольшой массив для проверки всех реализаций.

In [4]:
ys = [-0.5, 0.1, 0.7, -0.1]
hs_ref = np.fft.fft(ys)
hs_ref

array([ 0.2+0.j , -1.2-0.2j,  0.2+0.j , -1.2+0.2j])

In [5]:
hs_dft = dft(ys)
np.sum(np.abs(hs_ref - hs_dft))

np.float64(5.864775846765962e-16)

### Нерекурсивная версия (fft_norec)

Разбиваем сигнал на чётные и нечётные элементы, вычисляем их БПФ через `np.fft.fft`, затем объединяем через лемму Даниэльсона–Ланкзоса:

$$DFT(y)[n] = DFT(e)[n] + \exp(-2\pi i n / N) \cdot DFT(o)[n]$$

In [7]:
def fft_norec(ys):
    N = len(ys)
    He = np.fft.fft(ys[::2])
    Ho = np.fft.fft(ys[1::2])

    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)

    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [8]:
hs_norec = fft_norec(ys)
np.sum(np.abs(hs_ref - hs_norec))

np.float64(0.0)

### Рекурсивная версия (fft)

Заменяем `np.fft.fft` на рекурсивные вызовы. Базовый случай: при N=1 возвращаем исходный массив, так как ДПФ от одного отсчёта равен самому отсчёту.

In [9]:
def fft(ys):
    N = len(ys)
    if N == 1:
        return ys

    He = fft(ys[::2])
    Ho = fft(ys[1::2])

    ns = np.arange(N)
    W = np.exp(-1j * PI2 * ns / N)

    return np.tile(He, 2) + W * np.tile(Ho, 2)

In [10]:
hs_fft = fft(ys)
np.sum(np.abs(hs_ref - hs_fft))

np.float64(1.6653345369377348e-16)

### Проверка на реальном аудиосигнале

Загружаем WAV-файл, сравниваем результаты рекурсивного БПФ с `np.fft.fft`.

In [11]:
from thinkdsp import read_wave

wave = read_wave('../code/92002__jcveliz__violin-origional.wav')
ys_segment = wave.ys[:4096]
hs_np = np.fft.fft(ys_segment)
hs_mine = fft(ys_segment)

np.sum(np.abs(hs_np - hs_mine))

np.float64(1.1699538643046059e-13)